In [1]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")
doc_generator = (doc.text for doc in dataset.docs_iter())
print("docs generator created!")

docs generator created!


### Lemmatization

In [2]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import nltk
nltk.download("punkt_tab")
nltk.download("wordnet")

doc_generator = (doc.text for doc in dataset.docs_iter())
lemmatized_docs = []
lematizer = WordNetLemmatizer()

for text in doc_generator:
    tokens = word_tokenize(text)
    lemmatized_words = [lematizer.lemmatize(word) for word in tokens]
    lemmatized_docs.append(" ".join(lemmatized_words))

print("The texts lematized!")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tahas44/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/tahas44/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


The texts lematized!


In [23]:
lemmatized_docs

['it wa used in landing craft during world war ii and is used today in private boat and training facility the 6 71 is an inline six cylinder diesel engine the 71 refers to the displacement in cubic inch of each cylinder the firing order of the engine is 1 5 3 6 2 4 the engine s compression ratio is 18 7 1 with a 4 250 inch bore and a 5 00 inch stroke the engine weighs and is 54 inch long 29 inch wide and 41 inch tall at 2 100 revolution per minute the engine is capable of producing 230 horse power 172 kilowatt v type version of the 71 series were developed in 1957 the 6 71 is a two stroke engine a the engine will not naturally aspirate air is provided via a root type blower however on the 6 71t model a turbocharger and a supercharger are utilized fuel is provided by unit injector one per cylinder the amount of fuel injected into the engine is controlled by the engine s governor the engine cooling is via liquid in a water jacket in a boat cool external water is pumped into the engine',


In [24]:
len(lemmatized_docs)

369721

In [7]:
doc_generator = (doc.text for doc in dataset.docs_iter())
print("docs generator created!")

docs generator created!


In [12]:
lemmatized_docs[:100]

['after rejecting an offer from cambridge university she moved to london in 1954 working in a mayfair advertising agency while moonlighting a a hat check girl in the night club le club contemporain while working at the royal college of art she met the painter frank bowling when he wa still a student there they married in 1960 and had one son kitchen wa one of the woman interviewed by nell dunn in talking to woman 1965 after divorcing bowling in the late sixty kitchen went on to live with and later marry the writer dulan barber continuing to write novel she also began writing non fiction with biography of patrick geddes and gerard manley hopkins in later life she bought a house in barnwell northamptonshire which became the subject of her book of the same name she died on 23 november 2005 the novelist bessie head wa a close friend the pair corresponded from 1969 until head s death in 1986 on a range of subject including head s novel a question of power',
 'mat zan coached kuala lumpur fa

### CountVectorizer

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(stop_words="english")
bag_of_words = count_vectorizer.fit_transform(lemmatized_docs)
print("Bag of Words ready!")

Bag of Words ready!


In [26]:
bag_of_words

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 30625612 stored elements and shape (369721, 772400)>

In [4]:
queries = [query.text for query in dataset.queries_iter()]

lemmatized_queries = []

for query in queries:
    tokens = word_tokenize(query)
    lemmatized_words = [lematizer.lemmatize(word) for word in tokens]
    lemmatized_queries.append(" ".join(lemmatized_words))

query_vectors = count_vectorizer.transform(lemmatized_queries)

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(query_vectors, bag_of_words)

In [6]:
similarities

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.08944272, 0.02493773, ..., 0.        , 0.2534897 ,
        0.        ]], shape=(1444, 369721))

- necessary dictionaries

In [7]:
from collections import defaultdict
from helper import Scoredoc

doc_dict = defaultdict(str)

for i, doc in enumerate(dataset.docs_iter()):
    doc_dict[i] = doc.doc_id

doc_dict = dict(doc_dict)

qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

score_doc_dict = dict(score_doc_dict)

print("necessary dicts ready!")

necessary dicts ready!


In [8]:
import pandas as pd
query_ids = [query.query_id for query in dataset.queries_iter()]
df = pd.DataFrame(query_ids, columns=["Query_ID"])
df

,Query_ID
0,123839
1,188629
2,13898
3,316959
4,515031
...,...
1439,896124
1440,12319
1441,4421
1442,296526


In [9]:
from helper import create_AP, create_ndcg, create_statistical_columns, print_columns
df = create_statistical_columns(df, qrels_dict, doc_dict, similarities)
df = create_AP(df, qrels_dict, doc_dict, similarities)
df = create_ndcg(df, doc_dict, similarities, score_doc_dict)

print_columns(df)

recall_5_mean: 10.255333476210467
recall_5_std: 12.462686929921887
recall_5_max: 83.33333333333334
recall_5_min: 0.0
recall_10_mean: 14.343922728740486
recall_10_std: 16.743586141703233
recall_10_max: 100.0
recall_10_min: 0.0
precision_5_mean: 22.022160664819946
precision_5_std: 21.35557287046146
precision_5_max: 100.0
precision_5_min: 0.0
precision_10_mean: 16.024930747922436
precision_10_std: 15.934357342906702
precision_10_max: 90.0
precision_10_min: 0.0
f_score_5_mean: 12.900032241144654
f_score_5_std: 14.567865086217664
f_score_5_max: 90.9090909090909
f_score_5_min: 0.0
f_score_10_mean: 13.500194191223686
f_score_10_std: 14.402714824219604
f_score_10_max: 88.88888888888889
f_score_10_min: 0.0
MAP_5: 0.08062775019483055
MAP_10: 0.09666268133713661
NDCG_5_mean: 0.43219771029825593
NDCG_5_std: 0.342658667664897
NDCG_5_max: 1.0000000000000002
NDCG_5_min: 0.0
NDCG_10_mean: 0.43041741121897675
NDCG_10_std: 0.3394838071662572
NDCG_10_max: 1.0000000000000002
NDCG_10_min: 0.0
